In [3]:
from pathlib import Path
import torch

from datasets import load_dataset
import torch.nn as nn

import tokenizers
import logging


In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)

def enable_logging():
    logging.disable(logging.NOTSET)


def disable_logging():
    logging.disable(logging.DEBUG)


enable_logging()

In [4]:
nmt_original_valid_set, nmt_test_set = load_dataset(
    path="ageron/tatoeba_mt_train", name="eng-spa",
    split=["validation", "test"])
split = nmt_original_valid_set.train_test_split(train_size=0.8, seed=42)
nmt_train_set, nmt_valid_set = split["train"], split["test"]

2026-11-18 21:09:32,509 | INFO | httpx2 | HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-11-18 21:09:32,765 | INFO | httpx2 | HTTP Request: HEAD https://huggingface.co/datasets/ageron/tatoeba_mt_train/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-11-18 21:09:33,006 | INFO | httpx2 | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/ageron/tatoeba_mt_train/9b91de5e513db8bfa0c9d6bcf60efdc69a2a73d8/README.md?%2Fdatasets%2Fageron%2Ftatoeba_mt_train%2Fresolve%2Fmain%2FREADME.md=&etag=%2240c8b07584f0d4286989b716070e36d80a89eb8d%22 "HTTP/1.1 200 OK"
2026-11-18 21:09:33,260 | INFO | httpx2 | HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/ageron/tatoeba_mt_train/9b91de5e513db8bfa0c9d6bcf60efdc69a2a73d8/README.md?%2Fdatasets%2Fageron%2Ftatoeba_mt_train%2Fresolve%2Fmain%2FREADME.md=&etag=%2240c8b07584f0d4286989b716070e36d80a89eb8d%22 "HTTP/1.1 200 OK"
C:\Projects\Exp\RNN\.venv\Lib\site-packages\huggingfac

In [20]:
def get_data():
    for data in nmt_train_set:
        yield data["source_text"], data["target_text"]

    for data in nmt_test_set:
        yield data["source_text"], data["target_text"]

tokenizer_path = Path("tokenziers") / "NMT.json"
tokenizer = tokenizers.Tokenizer(tokenizers.models.BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.ByteLevel()
tokenizer.decoder = tokenizers.decoders.ByteLevel()

if tokenizer_path.exists():
    logging.info(f"Found existing tokenzier. Loading from {tokenizer_path}!")
    tokenizer = tokenizers.Tokenizer.from_file(str(tokenizer_path))
else:
    logging.info(f"not found existing tokenzier. saving to {tokenizer_path}!")
    token_trainer = tokenizers.trainers.BpeTrainer(
        vocab_size=1000, special_tokens=["[PAD]", "[UNK]", "[MASK]", "[CLS]", "[SEP]", "[SOS]", "[EOS]"]
    )

    tokenizer.train_from_iterator(get_data(),token_trainer)

    tokenizer.save(str(tokenizer_path))

tokenizer.enable_padding(pad_id=tokenizer.token_to_id("[PAD]"), pad_token="[PAD]")

print("Vocab size:", tokenizer.get_vocab_size())
print("PAD ID:", tokenizer.token_to_id("[PAD]"))
print("UNK ID:", tokenizer.token_to_id("[UNK]"))

encoded = tokenizer.encode('Tom tried to break up the fight.')

print("Tokens:", encoded.tokens)
print("IDs:", encoded.ids)
print("Decoded:", tokenizer.decode(encoded.ids))

2026-11-18 21:43:43,714 | INFO | root | Found existing tokenzier. Loading from tokenziers\NMT.json!


Vocab size: 1000
PAD ID: 0
UNK ID: 1
Tokens: ['ĠTom', 'Ġtr', 'i', 'ed', 'Ġto', 'Ġb', 're', 'ak', 'Ġup', 'Ġthe', 'Ġf', 'ight', '.']
IDs: [211, 353, 75, 202, 204, 196, 193, 464, 540, 209, 200, 447, 20]
Decoded:  Tom tried to break up the fight.


In [16]:
class NMTRNN(nn.Module):
    def __init__(self, vocab_size:int,hidden_size:int = 64, gru_num_layers:int = 1, embedding_dim:int=32, padding_idx:int = 0):
        super().__init__()
        
        self.embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim,padding_idx=padding_idx)

        self.encoder = nn.GRU(
            input_size=embedding_dim, 
            hidden_size=hidden_size, 
            num_layers = gru_num_layers, 
            batch_first=True
        )

        self.decoder = nn.GRU(
            input_size=hidden_size, 
            hidden_size=hidden_size, 
            num_layers = gru_num_layers, 
            batch_first=True
        )

        self.classifier = nn.Linear(in_features=hidden_size, out_features=vocab_size)


    def forward(self, X1, X2):
        logging.debug(f"X1: {X1.shape}")
        logging.debug(f"X2: {X2.shape}")
        
        input_embeddings = self.embedding_layer(X1)
        logging.debug(f"input_embeddings: {input_embeddings.shape}")
        encoder_output, _ = self.encoder(input_embeddings)
        logging.debug(f"encoder_output: {encoder_output.shape}")

        output_embeddings = self.embedding_layer(X2)
        logging.debug(f"output_embeddings: {output_embeddings.shape}")
        decoder_output, _ = self.decoder(encoder_output)
        logging.debug(f"decoder_output: {decoder_output.shape}")
        
        output = self.classifier(decoder_output)
        logging.debug(f"output: {output.shape}")
        output = output.permute(0, 2, 1)
        logging.debug(f"permutted output: {output.shape}")
        return output
        

{'source_text': 'Tom tried to break up the fight.',
 'target_text': 'Tom trató de disolver la pelea.',
 'source_lang': 'eng',
 'target_lang': 'spa'}